In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import ipywidgets as widgets
from ipywidgets import interact

def gut_microbiome_model(t, y, params):
    B, E, M, a, h, p = y  # Variables: bacterias y metabolitos
    q, V, Q, mu_B, mu_Ea, mu_Ep, mu_Ma, mu_Mh, Y_B, Y_E, Y_M, P_in = params

    # Términos de crecimiento con la cinética de Monod
    mu_B_p = mu_B * p / (p + 1)  # Crecimiento de B. thetaiotaomicron con polisacáridos
    mu_E_a = mu_Ea * a / (a + 1)  # Crecimiento de E. rectale con acetato
    mu_E_p = mu_Ep * p / (p + 1)  # Crecimiento de E. rectale con polisacáridos
    mu_M_a = mu_Ma * a / (a + 1)  # Crecimiento de M. smithii con acetato
    mu_M_h = mu_Mh * h / (h + 1)  # Crecimiento de M. smithii con H₂ + CO₂

    # Ecuaciones diferenciales
    dB_dt = (mu_B_p * B) - q * B
    dE_dt = (mu_E_a * E + mu_E_p * E) - q * E
    dM_dt = (mu_M_a * M + mu_M_h * M) - q * M
    da_dt = (Y_B * mu_B_p * B) - (Y_E * mu_E_a * E) - (Y_M * mu_M_a * M) - q * a
    dh_dt = (Y_E * mu_E_p * E) - (Y_M * mu_M_h * M) - q * h
    dp_dt = P_in - (mu_B_p * B + mu_E_p * E) - q * p  # Entrada de polisacáridos y consumo

    return [dB_dt, dE_dt, dM_dt, da_dt, dh_dt, dp_dt]

def simulate_model(q=0.1, mu_B=0.5, mu_Ea=0.3, mu_Ep=0.2, mu_Ma=0.4, mu_Mh=0.3, P_in=0.5):
    params = [q, 1, 1, mu_B, mu_Ea, mu_Ep, mu_Ma, mu_Mh, 0.2, 0.3, 0.1, P_in]

    # Condiciones iniciales
    y0 = [0.1, 0.05, 0.02, 0.3, 0.2, 1.0]  # B, E, M, Acetato, H2+CO2, Polisacáridos
    t_span = (0, 48)  # 48 horas
    t_eval = np.linspace(0, 48, 200)

    # Resolver el sistema de ecuaciones diferenciales
    sol = solve_ivp(gut_microbiome_model, t_span, y0, args=(params,), t_eval=t_eval)

    # Graficar resultados
    plt.figure(figsize=(10, 5))
    plt.plot(sol.t, sol.y[0], label='B. thetaiotaomicron', color='blue')
    plt.plot(sol.t, sol.y[1], label='E. rectale', color='green')
    plt.plot(sol.t, sol.y[2], label='M. smithii', color='red')
    plt.plot(sol.t, sol.y[3], label='Acetato', linestyle='dashed', color='purple')
    plt.plot(sol.t, sol.y[4], label='CO2 + H2', linestyle='dashed', color='orange')
    plt.plot(sol.t, sol.y[5], label='Polisacáridos', linestyle='dashed', color='brown')

    plt.xlabel('Tiempo (horas)')
    plt.ylabel('Concentración')
    plt.legend()
    plt.title('Simulación del modelo del microbioma intestinal')
    plt.show()

# Interfaz interactiva
interact(simulate_model, q=(0.05, 0.2, 0.01), mu_B=(0.1, 1.0, 0.1), mu_Ea=(0.1, 1.0, 0.1),
         mu_Ep=(0.1, 1.0, 0.1), mu_Ma=(0.1, 1.0, 0.1), mu_Mh=(0.1, 1.0, 0.1), P_in=(0.1, 1.0, 0.1));


interactive(children=(FloatSlider(value=0.1, description='q', max=0.2, min=0.05, step=0.01), FloatSlider(value…